# Evaluation with LangSmith & RAGAS

1. **Challenges in RAG Evaluation** - Understanding the complexity
2. **LangSmith** - Observability and tracking platform
3. **RAGAS** - Specific metrics for RAG systems
4. **A/B Testing** - Comparing production versions
5. **Regression Testing** - Ensuring continuous quality

---

## 📦 Installation and Imports

In [1]:

!pip install langchain langchain-google-genai langchain-community chromadb tiktoken ragas datasets


### Importing Libraries

In [2]:
# Necessary imports
import os
import pandas as pd
import numpy as np
from datetime import datetime
import json
import time
import warnings
from dotenv import load_dotenv
from pathlib import Path  # Added for compatibility with Path
from typing import Union  # Added for backward compatibility with type annotations
warnings.filterwarnings('ignore')
print("✅ Standard libraries imported successfully!")


✅ Standard libraries imported successfully!


In [3]:
# LangChain imports
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.vectorstores import Chroma
from langchain.schema import Document
from langchain.text_splitter import CharacterTextSplitter
from langchain.chat_models import ChatOpenAI
from langchain.embeddings.openai import OpenAIEmbeddings

print("✅ LangChain libraries imported successfully!")

✅ LangChain libraries imported successfully!


In [5]:
# RAGAS imports
import sys
!{sys.executable} -m pip install eval_type_backport

from ragas import evaluate
from datasets import Dataset
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)


print("✅ RAGAS libraries imported successfully!")

Defaulting to user installation because normal site-packages is not writeable
  Using cached eval_type_backport-0.2.2-py3-none-any.whl.metadata (2.2 kB)
Using cached eval_type_backport-0.2.2-py3-none-any.whl (5.8 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
✅ RAGAS libraries imported successfully!


## 🔧 Initial Setup

Let's configure the necessary API keys.

In [6]:

load_dotenv(dotenv_path='../../.env')  # Specify the path to your .env file

# Access the environment variable
api_key = os.getenv('OPENAI_API_KEY')

# Check if the variable is loaded
if api_key or api_key == "":
    print(f"API key loaded successfully.")
else:
    print("Failed to load API key.")

from openai import OpenAI
client = OpenAI(api_key=api_key)

API key loaded successfully.


## 📚 1. Preparation of Test Data

Let's create a test dataset to demonstrate the evaluation metrics.

In [7]:
# Creation of example documents on AI and technology
knowledge_documents = [
    """Artificial Intelligence (AI) is a field of computer science focused on creating systems capable of performing tasks that typically require human intelligence. This includes learning, reasoning, perception, and decision-making. AI can be classified into weak AI (task-specific) and strong AI (general intelligence).""",

    """Machine Learning is a subfield of AI that enables computers to learn and improve automatically through experience without being explicitly programmed. ML algorithms identify patterns in data and make predictions. There are three main types: supervised learning, unsupervised learning, and reinforcement learning.""",

    """Deep Learning is a machine learning technique based on artificial neural networks with multiple layers. It is particularly effective for tasks such as image recognition, natural language processing, and speech recognition. Deep neural networks can have hundreds of layers and millions of parameters.""",

    """RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. It allows language models to access external knowledge to generate more accurate and up-to-date responses. The process involves retrieving relevant documents and using this information to generate the final response.""",

    """Google Gemini is a multimodal language model developed by Google, capable of processing text, images, and code. It offers advanced reasoning and contextual understanding capabilities. Gemini comes in different versions: Nano, Pro, and Ultra, each optimized for different use cases.""",

    """LangChain is a framework for developing applications with language models. It facilitates the creation of complex chains, memory management, and integration with various data sources. It offers modular components to build robust conversational AI applications."""
]

# Conversion to Document objects
docs = [Document(page_content=doc) for doc in knowledge_documents]

print(f"✅ Created {len(docs)} knowledge documents")

✅ Created 6 knowledge documents


In [8]:
# Creation of test dataset for RAGAS evaluation
test_data = {
    'question': [
        "What is Artificial Intelligence?",
        "How does Machine Learning work?",
        "What are the applications of Deep Learning?",
        "What is RAG and how does it work?",
        "What are the features of Google Gemini?"
    ],
    'ground_truth': [
        "Artificial Intelligence is a field of computer science focused on creating systems that perform tasks requiring human intelligence, including learning, reasoning, and decision-making.",
        "Machine Learning enables computers to learn automatically through experience, identifying patterns in data to make predictions without explicit programming.",
        "Deep Learning is effective for image recognition, natural language processing, and speech recognition, using neural networks with multiple layers.",
        "RAG combines information retrieval with text generation, allowing models to access external knowledge for more accurate responses.",
        "Google Gemini is a multimodal model that processes text, images, and code, offering advanced reasoning capabilities in Nano, Pro, and Ultra versions."
    ]
}

print("✅ Test dataset created!")
print(f"📊 {len(test_data['question'])} test questions prepared")

✅ Test dataset created!
📊 5 test questions prepared


## 🔍 2. RAG


In [9]:

embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    api_key=api_key
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db_avaliacao"
)

llm = ChatOpenAI(
    model= "gpt-4o-mini",
    openai_api_key=api_key,
    temperature=0.3,
)

memory= ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=3,
    return_messages=True,
    output_key='answer'
)

rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    return_source_documents=True,
    verbose=False
)

print("✅ RAG created!")

/var/folders/v3/j1dtbpl50r9_jp0s85h7c3lw0000gp/T/ipykernel_84971/2944054268.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(


✅ RAG created!


/var/folders/v3/j1dtbpl50r9_jp0s85h7c3lw0000gp/T/ipykernel_84971/2944054268.py:12: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(
/var/folders/v3/j1dtbpl50r9_jp0s85h7c3lw0000gp/T/ipykernel_84971/2944054268.py:18: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory= ConversationBufferWindowMemory(


## 🧪 3. Data Collection for RAGAS Evaluation

In [10]:
def execute_rag_and_collect_data(questions):
    """Executes the RAG system and collects data for evaluation"""

    results = {
        'question': [],
        'answer': [],
        'contexts': [],
        'ground_truth': []
    }

    for i, question in enumerate(questions):
        print(f"\n🔄 Processing question {i+1}/{len(questions)}: {question}")

        try:
            # Execute RAG
            result = rag_chain({"question": question})

            # Extract contexts from source documents
            contexts = [doc.page_content for doc in result['source_documents']]

            # Store results
            results['question'].append(question)
            results['answer'].append(result['answer'])
            results['contexts'].append(contexts)
            results['ground_truth'].append(test_data['ground_truth'][i])

            print(f"✅ Generated answer: {result['answer'][:100]}...")

        except Exception as e:
            print(f"❌ Error processing question: {str(e)}")
            continue

    return results

# Execute data collection
print("🚀 Starting data collection for evaluation...")
evaluation_data = execute_rag_and_collect_data(test_data['question'])

print(f"\n✅ Data collection completed! {len(evaluation_data['question'])} examples collected")


🚀 Starting data collection for evaluation...

🔄 Processing question 1/5: What is Artificial Intelligence?


/var/folders/v3/j1dtbpl50r9_jp0s85h7c3lw0000gp/T/ipykernel_84971/1482873859.py:16: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = rag_chain({"question": question})


✅ Generated answer: Artificial Intelligence (AI) is a field of computer science focused on creating systems capable of p...

🔄 Processing question 2/5: How does Machine Learning work?
✅ Generated answer: Machine Learning works by enabling computers to learn from data and improve their performance over t...

🔄 Processing question 3/5: What are the applications of Deep Learning?
✅ Generated answer: Deep Learning is particularly effective for tasks such as image recognition, natural language proces...

🔄 Processing question 4/5: What is RAG and how does it work?
✅ Generated answer: RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text ge...

🔄 Processing question 5/5: What are the features of Google Gemini?
✅ Generated answer: Google Gemini is a multimodal language model that can process text, images, and code. It offers adva...

✅ Data collection completed! 5 examples collected


## 📊 4. Evaluation with RAGAS


In [11]:

dataset_ragas = Dataset.from_dict(evaluation_data)

print("📋 Dataset ready for evaluation with RAGAS:")
print(f"   - {len(dataset_ragas)} examples")
print(f"   - Columns: {list(dataset_ragas.column_names)}")

metrics_ragas = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

print("\n🎯 Configured RAGAS metrics:")
for metric in metrics_ragas:
    print(f"   - {metric.name}")

📋 Dataset ready for evaluation with RAGAS:
   - 5 examples
   - Columns: ['question', 'answer', 'contexts', 'ground_truth']

🎯 Configured RAGAS metrics:
   - faithfulness
   - answer_relevancy
   - context_precision
   - context_recall


In [12]:
# Execute RAGAS evaluation
print("🔍 Starting evaluation with RAGAS...")
print("⏳ This may take a few minutes...")

# Configure LLM for RAGAS (using OpenAI)
ragas_result = evaluate(
    dataset_ragas,
    metrics=metrics_ragas,
    llm=llm,
    embeddings=embeddings
)

print("\n✅ RAGAS evaluation completed!")

# Display results
print("\n📊 === RAGAS EVALUATION RESULTS ===")

# Access individual metrics from the EvaluationResult object
print(f"faithfulness: {ragas_result['faithfulness'][0]:.4f}")
print(f"answer_relevancy: {ragas_result['answer_relevancy'][0]:.4f}")
print(f"context_precision: {ragas_result['context_precision'][0]:.4f}")
print(f"context_recall: {ragas_result['context_recall'][0]:.4f}")

print("💡 Let's create a simulated evaluation for demonstration...")

# Simulated RAGAS results for demonstration
simulated_ragas_result = {
    'faithfulness': 0.85,
    'answer_relevancy': 0.78,
    'context_relevancy': 0.82,
    'context_recall': 0.75
}

print("\n📊 === SIMULATED RAGAS EVALUATION RESULTS ===")
for metric, value in simulated_ragas_result.items():
    print(f"{metric}: {value:.4f}")

🔍 Starting evaluation with RAGAS...
⏳ This may take a few minutes...


Evaluating:   5%|▌         | 1/20 [00:03<01:00,  3.17s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 20/20 [00:43<00:00,  2.15s/it]



✅ RAGAS evaluation completed!

📊 === RAGAS EVALUATION RESULTS ===
faithfulness: 1.0000
answer_relevancy: 0.9319
context_precision: 1.0000
context_recall: 1.0000
💡 Let's create a simulated evaluation for demonstration...

📊 === SIMULATED RAGAS EVALUATION RESULTS ===
faithfulness: 0.8500
answer_relevancy: 0.7800
context_relevancy: 0.8200
context_recall: 0.7500


## 📈 5. Detailed Analysis of Metrics

In [13]:
def analyze_ragas_metrics(results):
    """Analyzes and interprets a RAGAS EvaluationResult (or dict),
    printing each metric with qualitative insights."""

    # Convert the EvaluationResult object to a simple dictionary
    if hasattr(results, "to_dict"):                     # RAGAS ≥ 0.0.12
        results = results.to_dict()
    elif hasattr(results, "_scores_dict"):              # RAGAS 0.0.10 – 0.0.11
        results = results._scores_dict
    elif not isinstance(results, dict):                 # Any remaining Mapping
        results = dict(results)                         # Conversion fallback

    # Helper to safely retrieve the score as a scalar
    def get_score(key: str, default: float = 0.0) -> float:
        value = results.get(key, default)
        # Some backends return a list or ScoreList ➜ take the first item
        if isinstance(value, (list, tuple)):
            return float(value[0])
        return float(value)

    print("🔍 === DETAILED ANALYSIS OF RAGAS METRICS ===\n")

    # Expected metrics (some installations use context_precision
    # instead of context_relevancy ― we cover both)
    metrics = {
        "faithfulness":       "Faithfulness",
        "answer_relevancy":   "Answer Relevancy",
        "context_relevancy":  "Context Relevancy",
        "context_precision":  "Context Precision",
        "context_recall":     "Context Recall",
    }

    valid_scores = []
    for key, readable_name in metrics.items():
        if key not in results:
            continue                                  # Skip missing metrics
        score = get_score(key)
        valid_scores.append(score)

        # Display numeric value
        print(f"📊 **{readable_name}: {score:.4f}**")

        # Qualitative interpretation
        if score >= 0.80:
            print("   ✅ Excellent!")
        elif score >= 0.60:
            print("   ⚠️ Moderate.")
        else:
            print("   ❌ Low.")

    # Overall score (simple average of present scores)
    if valid_scores:
        overall_score = float(np.mean(valid_scores))
        print(f"\n🎯 **Overall Score: {overall_score:.4f}**")
        if overall_score >= 0.80:
            print("   🏆 RAG system with excellent performance!")
        elif overall_score >= 0.60:
            print("   👍 RAG system with good performance, but room for improvement")
        else:
            print("   🔧 RAG system needs significant optimization")
    else:
        print("⚠️ No metrics available for analysis.")


In [14]:
analyze_ragas_metrics(ragas_result)


🔍 === DETAILED ANALYSIS OF RAGAS METRICS ===

📊 **Faithfulness: 1.0000**
   ✅ Excellent!
📊 **Answer Relevancy: 0.9319**
   ✅ Excellent!
📊 **Context Precision: 1.0000**
   ✅ Excellent!
📊 **Context Recall: 1.0000**
   ✅ Excellent!

🎯 **Overall Score: 0.9830**
   🏆 RAG system with excellent performance!
